In [9]:
%pip install flask flask-cors

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
Using cached flask_cors-6.0.2-py3-none-any.whl (13 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import sqlite3
from flask import Flask, jsonify, request
from flask_cors import CORS

app = Flask(__name__)

# We need cors to be able to use the entire flask thing
CORS(app, resources={r"/api/*": {"origins": "*"}})

@app.route('/api/travel', methods=['GET'])
def get_travel_data():
    # Example of a search: /api/travel?country=Italy
    search_query = request.args.get('country') 
    
    conn = sqlite3.connect('travel_planner.db')
    conn.row_factory = sqlite3.Row 
    cursor = conn.cursor()
    
    # Modify the SQL based on whether the user searched for something
    if search_query:
        # The ? is for safety (prevents SQL injection)
        # The % signs mean "find this word anywhere in the country name"
        cursor.execute("SELECT id, country, capital, region FROM Country WHERE country LIKE ?", ('%' + search_query + '%',))
    else:
        # If they didn't search for anything, give them the whole database
        cursor.execute("SELECT id, country, capital, region FROM Country")
        
    countries = cursor.fetchall()
    
    # Fetch all cities (same as before)
    cursor.execute("SELECT city, country_id FROM Cities")
    cities = cursor.fetchall()
    
    cities_by_country = {}
    for city in cities:
        c_id = city['country_id']
        if c_id not in cities_by_country:
            cities_by_country[c_id] = []
        cities_by_country[c_id].append(city['city'])
        
    # Build the final JSON
    output = []
    for c in countries:
        output.append({
            "id": c['id'],
            "name": c['country'],
            "capital": c['capital'],
            "region": c['region'],
            "cities": cities_by_country.get(c['id'], [])
        })
        
    conn.close()
    return jsonify(output)

if __name__ == '__main__':
    print("Starting Flask server on http://127.0.0.1:5001")
    app.run(port=5001)

Starting Flask server on http://127.0.0.1:5001
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [04/May/2026 12:28:06] "GET /api/travel?country=italy HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 12:28:07] "GET /api/travel?country=italy HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 12:28:08] "GET /api/travel?country=italy HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 12:28:08] "GET /api/travel?country=italy HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 12:28:09] "GET /api/travel?country=italy HTTP/1.1" 200 -
